# Camera positions around the insect

Interactive 3D view of a photo session: every matched photo's camera position
(from `frames.csv`) as a dot **in the insect's body frame** (insect body
origin at the origin), colored by **camera height level**, with an **arrow
showing each camera's OptiTrack viewing direction**. Drag to rotate, scroll
to zoom, hover a dot for the photo it corresponds to.

Notes until the Stage 2 calibrations exist:
- positions/orientations are of the **phone marker body**, not the lens —
  expect a constant offset (hand-eye, Stage 2.4);
- the arrows rotate a **fixed phone-body axis** (`LOOK_AXIS`, set it to
  match how the body was defined in Motive) by each pose quaternion —
  purely what OptiTrack reports, independent of where the insect sits;
- `N_LEVELS` controls how many height levels the ring detection splits into.

**Kernel:** the pixi env interpreter (`.pixi/envs/default/bin/python`).
Set `SESSION` to the session you want to look at.

In [ ]:
import csv
from pathlib import Path

import numpy as np

SESSION = Path('../datasets/phone_20260709_100920')   # ← change per session
if not SESSION.exists():                               # also works from repo root
    SESSION = Path(str(SESSION).removeprefix('../'))

rows = list(csv.DictReader(open(SESSION / 'frames.csv')))
cams = np.array([[float(r['x']), float(r['y']), float(r['z'])] for r in rows])
quats = np.array([[float(r['qx']), float(r['qy']), float(r['qz']), float(r['qw'])]
                  for r in rows])
names = [Path(r['image_file']).name for r in rows]
gap_ms = [float(r['gap_ms']) for r in rows]
spread_mm = [float(r['pos_spread_mm']) for r in rows]

d = np.linalg.norm(cams, axis=1)
print(f'{len(rows)} camera positions from {SESSION / "frames.csv"}')
print(f'camera–origin distance: min {d.min():.3f} m · '
      f'median {np.median(d):.3f} m · max {d.max():.3f} m')

In [ ]:
def q_rot(q, v):
    """Rotate vector v by xyzw quaternion q."""
    x, y, z, w = q
    R = np.array([[1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
                  [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
                  [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)]])
    return R @ np.asarray(v, dtype=float)

# Viewing direction straight from the OptiTrack orientation: a FIXED axis of
# the phone rigid body, rotated by each pose quaternion — independent of
# where the insect sits. Which body axis the lens looks along depends on how
# the body was defined in Motive; set it here (hand-eye calibration,
# Stage 2.4, will measure it exactly).
LOOK_AXIS = np.array([0.0, -1.0, 0.0])   # ← phone-body axis of the lens

look_dirs = np.array([q_rot(q, LOOK_AXIS) for q in quats])
print(f'arrows show phone-body axis {LOOK_AXIS.tolist()} rotated by each '
      f'OptiTrack orientation')

In [ ]:
# Height levels: shots taken at the same camera height form one ring, but
# the heights are only roughly equal — split the sorted z values at the
# (N_LEVELS - 1) largest gaps. Check the printout; adjust N_LEVELS to what
# the session actually had.
N_LEVELS = 4   # ← set to the number of camera heights used in this session

order = np.argsort(cams[:, 2])
zs = cams[order, 2]
cuts = set(np.argsort(np.diff(zs))[-(N_LEVELS - 1):]) if N_LEVELS > 1 else set()
sorted_levels = np.zeros(len(zs), dtype=int)
for i in range(1, len(zs)):
    sorted_levels[i] = sorted_levels[i - 1] + (1 if (i - 1) in cuts else 0)
level = np.empty(len(cams), dtype=int)
level[order] = sorted_levels

for k in range(level.max() + 1):
    sel = level == k
    print(f'level {k + 1}: {sel.sum():2d} shots · '
          f'z {cams[sel, 2].min():.3f}–{cams[sel, 2].max():.3f} m '
          f'(mean {cams[sel, 2].mean():.3f})')

In [ ]:
import plotly.graph_objects as go

# categorical palette (fixed slot order) — one hue per height level;
# red is reserved for the insect marker
CAT = ['#2a78d6', '#1baf7a', '#eda100', '#008300', '#4a3aa7', '#e87ba4']

# viewing-direction arrows as line segments (camera → 4 cm along look axis)
ARROW_LEN = 0.04
tips = cams + ARROW_LEN * look_dirs
ax, ay, az = [], [], []
for c, t in zip(cams, tips):
    ax += [c[0], t[0], None]
    ay += [c[1], t[1], None]
    az += [c[2], t[2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(                       # faint path in shot order
    x=cams[:, 0], y=cams[:, 1], z=cams[:, 2], mode='lines',
    line=dict(color='#b5b3ad', width=2), name='shot order', hoverinfo='skip'))
fig.add_trace(go.Scatter3d(
    x=ax, y=ay, z=az, mode='lines',
    line=dict(color='#55524c', width=4), name='view direction',
    hoverinfo='skip'))
for k in range(level.max() + 1):                  # cameras, one trace per level
    sel = np.flatnonzero(level == k)
    fig.add_trace(go.Scatter3d(
        x=cams[sel, 0], y=cams[sel, 1], z=cams[sel, 2], mode='markers',
        name=f'level {k + 1} · z≈{cams[sel, 2].mean():.2f} m',
        marker=dict(size=8, color=CAT[k % len(CAT)]),
        customdata=[[names[i], i + 1, gap_ms[i], spread_mm[i]] for i in sel],
        hovertemplate=('<b>%{customdata[0]}</b> (shot #%{customdata[1]})<br>'
                       'x=%{x:.3f}  y=%{y:.3f}  z=%{z:.3f} m<br>'
                       'pose gap %{customdata[2]} ms · '
                       'motion ±50 ms %{customdata[3]} mm<extra></extra>')))
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers+text',
    marker=dict(size=10, color='#e34948', symbol='diamond'),
    text=['insect'], textposition='top center',
    textfont=dict(color='#55524c', size=13), name='insect (body origin)',
    hovertemplate='popillia body-frame origin<extra></extra>'))
fig.update_layout(
    title=f'Cameras in the insect body frame — {SESSION.name}',
    scene=dict(xaxis_title='x (m)', yaxis_title='y (m)', zaxis_title='z (m)',
               aspectmode='data'),                 # true metric proportions
    template='plotly_white', height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()